# Example 11.6.1 / 2: Production Equilibrium

Companion notebook for **11.6.1 Equilibrium Problems and MPCC** in the AMPL lab Documentation, section "Production Equilibrium". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Production Equilibrium

At equilibrium, a positive activity level has zero unit profit, while an activity with strictly negative unit profit is unused. A positive product price likewise requires output to meet demand exactly.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                       # products
set ACT;                        # production activities
param cost {ACT} > 0;           # unit cost of each activity
param demand {PROD} >= 0;       # demand per product
param output {PROD, ACT} >= 0;  # output of product i per unit of activity j

var Price {PROD};  # product price
var Level {ACT};   # activity level

subject to ProductMarket {i in PROD}:  # positive price only if output meets demand
    Price[i] >= 0 complements
    sum {j in ACT} output[i,j] * Level[j] >= demand[i];

subject to ActivityProfit {j in ACT}:  # positive level only at zero unit profit
    Level[j] >= 0 complements
    sum {i in PROD} Price[i] * output[i,j] <= cost[j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["goods"]
ampl.set["ACT"] = ["factory"]
ampl.param["cost"] = {"factory": 12}
ampl.param["demand"] = {"goods": 6}
ampl.param["output"] = {("goods", "factory"): 2}

In [5]:
# run.py
print(ampl.get_output("expand ProductMarket, ActivityProfit;"), end="")

subject to ProductMarket['goods']:
	0 <= Price['goods']
     complements
	6 <= 2*Level['factory'];

subject to ActivityProfit['factory']:
	0 <= Level['factory']
     complements
	2*Price['goods'] <= 12;



### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := goods;
set ACT := factory;
param cost := factory 12;
param demand := goods 6;
param output := goods factory 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
expand ProductMarket, ActivityProfit;

subject to ProductMarket['goods']:
	0 <= Price['goods']
     complements
	6 <= 2*Level['factory'];

subject to ActivityProfit['factory']:
	0 <= Level['factory']
     complements
	2*Price['goods'] <= 12;



### Another way in amplpy

For this example, equilibrium is `Level["factory"] = 3` and `Price["goods"] = 6`. That conclusion comes from both complementarity pairs, not from minimizing a cost objective.

Solving the same `model.mod` and `data.py` confirms it. HiGHS, through the MP conversion of
`complements`, returns the equilibrium. The model is square, with two variables and two
complementarity constraints (`_nvars` and `_nccons`; AMPL presolve may shrink the solver's view of
such a small system):

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["goods"]
ampl.set["ACT"] = ["factory"]
ampl.param["cost"] = {"factory": 12}
ampl.param["demand"] = {"goods": 6}
ampl.param["output"] = {("goods", "factory"): 2}

In [11]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["Price"].to_dict(), ampl.var["Level"].to_dict())
print([ampl.get_value(n) for n in ("_nvars", "_nccons")])

solved {'goods': 6} {'factory': 3}
[2, 2]
